# PCCoT Training on Colab Pro

This notebook sets up and runs PCCoT training on Google Colab Pro.

## Step 1: Check GPU and Mount Google Drive

In [ ]:
# Check GPU
!nvidia-smi

# Mount Google Drive (for saving models and data)
from google.colab import drive
drive.mount('/content/drive')

# Create directories
!mkdir -p /content/drive/MyDrive/PCCoT/outputs
!mkdir -p /content/drive/MyDrive/PCCoT/cache

## Step 2: Clone/Upload Repository

In [ ]:
# Option 1: If your code is on GitHub, clone it
# !git clone https://github.com/yourusername/PCCoT.git /content/PCCoT

# Option 2: Upload your code via Colab file browser (Files -> Upload)
# Then extract if it's a zip file:
# !unzip -q /content/PCCoT.zip -d /content/

# For now, assuming you'll upload the code
# Change to the directory (adjust path if needed)
%cd /content
# If you uploaded a zip, uncomment:
# !unzip -q PCCoT.zip
%cd PCCoT

## Step 3: Install Dependencies

In [ ]:
# Install PyTorch with CUDA support (Colab usually has CUDA 12.1)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Install core dependencies (with NumPy < 2.0 to avoid compatibility issues)
!pip install "numpy<1.28" scipy pandas numexpr bottleneck

# Install transformers and related packages
!pip install transformers accelerate datasets evaluate peft bitsandbytes

# Install flash attention (for faster training)
!pip install flash-attn --no-build-isolation

# Install other dependencies if needed
!pip install scikit-learn

## Step 4: Set Environment Variables

In [ ]:
import os

# Set HuggingFace endpoint (if using mirror)
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

# Set cache directory to Drive (to persist between sessions)
os.environ['HF_HOME'] = '/content/drive/MyDrive/PCCoT/cache'
os.environ['TRANSFORMERS_CACHE'] = '/content/drive/MyDrive/PCCoT/cache'

# Disable user site-packages to avoid conflicts
os.environ['PYTHONNOUSERSITE'] = '1'

## Step 5: Run Training (Adjusted for Colab GPU)

In [ ]:
# Training command - adjusted batch size for Colab GPU (usually T4 with 16GB VRAM)
# If you get OOM errors, reduce per_device_train_batch_size further

!python run_ccot.py \
    --model_name_or_path gpt2 \
    --config_name configs/pccot_gpt2_small.json \
    --config_overrides num_iterations=3 \
    --num_latent_tokens 24 \
    --dataset_name whynlp/gsm8k-aug \
    --label_names labels cot_labels \
    --lora_target_modules c_attn-c_fc-c_proj \
    --lora_modules_to_save "" \
    --remove_unused_columns false \
    --per_device_train_batch_size 64 \
    --per_device_eval_batch_size 16 \
    --gradient_accumulation_steps 2 \
    --block_size 1024 \
    --attn_implementation flash_attention_2 \
    --lr_scheduler_type cosine \
    --warmup_ratio 0.03 \
    --learning_rate 3e-3 \
    --weight_decay 1e-2 \
    --bf16 \
    --torch_dtype bfloat16 \
    --do_train \
    --do_eval \
    --do_predict \
    --num_train_epochs 40 \
    --save_total_limit 1 \
    --save_strategy steps \
    --save_steps 500 \
    --evaluation_strategy steps \
    --eval_steps 500 \
    --logging_steps 50 \
    --load_best_model_at_end True \
    --metric_for_best_model eval_ccot_exact_match \
    --report_to none \
    --run_name pccot-gpt2-lora-3-24-colab \
    --output_dir /content/drive/MyDrive/PCCoT/outputs/pccot-gpt2-lora-3-24

In [ ]:
# 保存训练 loss 和 eval 结果到 CSV

import os
import re
import json
import pandas as pd

# 和训练时的 --output_dir 保持一致（本地就改成你本地的 output_dir）
output_dir = "/content/drive/MyDrive/PCCoT/outputs/pccot-gpt2-lora-3-24"

# ===== 1）找到最新的 checkpoint 目录，并读取里面的 trainer_state.json =====
ckpt_dirs = [
    d for d in os.listdir(output_dir)
    if os.path.isdir(os.path.join(output_dir, d)) and d.startswith("checkpoint-")
]
if not ckpt_dirs:
    raise FileNotFoundError(f"No checkpoint-* directories found in {output_dir}")

# 取 global_step 最大的那个 checkpoint
step_pattern = re.compile(r"checkpoint-(\d+)")
ckpt_steps = []
for d in ckpt_dirs:
    m = step_pattern.match(d)
    if m:
        ckpt_steps.append((int(m.group(1)), d))

if not ckpt_steps:
    raise ValueError(f"No valid checkpoint-* with step number found in {output_dir}")

ckpt_steps.sort()
latest_step, latest_ckpt = ckpt_steps[-1]
latest_ckpt_dir = os.path.join(output_dir, latest_ckpt)
print(f"Using latest checkpoint: {latest_ckpt_dir} (step {latest_step})")

state_path = os.path.join(latest_ckpt_dir, "trainer_state.json")
if not os.path.exists(state_path):
    raise FileNotFoundError(f"trainer_state.json not found in {latest_ckpt_dir}")

with open(state_path, "r") as f:
    state = json.load(f)

log_history = state.get("log_history", [])

# 保存成 CSV，里面会有 step、loss、eval_xxx 等列
logs_csv_path = os.path.join(output_dir, "log_history.csv")
pd.DataFrame(log_history).to_csv(logs_csv_path, index=False)
print(f"Saved log history to {logs_csv_path}")

# ===== 2）eval 的汇总结果（最后一次）通常在 output_dir/all_results.json 里 =====
all_results_path = os.path.join(output_dir, "all_results.json")
if os.path.exists(all_results_path):
    with open(all_results_path, "r") as f:
        all_results = json.load(f)
    eval_csv_path = os.path.join(output_dir, "all_results.csv")
    pd.DataFrame([all_results]).to_csv(eval_csv_path, index=False)
    print(f"Saved eval summary to {eval_csv_path}")
else:
    print(f"No all_results.json found in {output_dir}")